In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/models/justineleonuro/deepseek-math/transformers/deepseek-math-7b-instruct/1/deepseek-math-7bin-p100/model.safetensors.index.json
/kaggle/input/models/justineleonuro/deepseek-math/transformers/deepseek-math-7b-instruct/1/deepseek-math-7bin-p100/model-00003-of-00003.safetensors
/kaggle/input/models/justineleonuro/deepseek-math/transformers/deepseek-math-7b-instruct/1/deepseek-math-7bin-p100/config.json
/kaggle/input/models/justineleonuro/deepseek-math/transformers/deepseek-math-7b-instruct/1/deepseek-math-7bin-p100/tokenizer.json
/kaggle/input/models/justineleonuro/deepseek-math/transformers/deepseek-math-7b-instruct/1/deepseek-math-7bin-p100/model-00001-of-00003.safetensors
/kaggle/input/models/justineleonuro/deepseek-math/transformers/deepseek-math-7b-instruct/1/deepseek-math-7bin-p100/tokenizer_config.json
/kaggle/input/models/justineleonuro/deepseek-math/transformers/deepseek-math-7b-instruct/1/deepseek-math-7bin-p100/model-00002-of-00003.safetensors
/kaggle/input/mode

In [7]:
import os
import re
import sympy as sp
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [18]:
# --- Configuration (you can tune these later) ---
COMPETITION_DIR_CANDIDATES = [
    "/kaggle/input/ai-mathematical-olympiad-progress-prize-3",
    "/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3",
]

ANSWER_MOD = 100000
DEBUG = True
N_SAMPLES = 3  # number of LLM reasoning paths per problem (self‑consistency)

# --- Resolve competition directory ---
def resolve_competition_dir():
    for path in COMPETITION_DIR_CANDIDATES:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(
        "Competition input directory was not found. "
        "Please check the actual path under /kaggle/input."
    )

competition_dir = resolve_competition_dir()

if DEBUG:
    print("Competition directory:", competition_dir)
    print("Files:", os.listdir(competition_dir))

test_path = os.path.join(competition_dir, "test.csv")
sample_submission_path = os.path.join(competition_dir, "sample_submission.csv")
test_df = pd.read_csv(test_path)
reference_df = pd.read_csv(sample_submission_path)

if DEBUG:
    print("test.csv shape:", test_df.shape)
    print("reference.csv shape:", reference_df.shape)
    print(test_df.head())

Competition directory: /kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3
Files: ['reference.csv', 'AIMO3_Reference_Problems.pdf', 'sample_submission.csv', 'test.csv', 'kaggle_evaluation']
test.csv shape: (3, 2)
reference.csv shape: (3, 2)
       id                 problem
0  000aaa          What is $1-1$?
1  111bbb    What is $0\times10$?
2  222ccc  Solve $4+x=4$ for $x$.


In [9]:
# --- Answer extraction from Latex-style \boxed{...} ---
def extract_boxed_number(text: str):
    if not isinstance(text, str):
        return None
    
    # Match \boxed{123} in raw Latex (note: Kaggle may double-escape backslashes)
    match = re.search(r'\\\\boxed\\{(-?\\d+)\\}', text)
    if match:
        return int(match.group(1)) % ANSWER_MOD
    
    return None

# --- Very simple sympy solver for easy polynomial equations ---
def solve_with_sympy_if_easy(problem: str):
    if not isinstance(problem, str):
        return 0
    
    # Conservative heuristic: only try very simple one‑variable equations
    try:
        cleaned = problem.strip()

        # Example pattern: "Solve x^2 - 5x + 6 = 0"
        simple_eq_match = re.search(r'([xX][^=]*?)=\\s*0', cleaned)
        if simple_eq_match and len(cleaned) < 200:
            x = sp.symbols('x')
            expr_text = simple_eq_match.group(1)

            # Normalize math notation
            expr_text = expr_text.replace("^", "**")
            expr_text = re.sub(r'(\\d)([xX])', r'\1*\2', expr_text)
            expr_text = expr_text.replace("X", "x")

            expr = sp.sympify(expr_text)
            roots = sp.solve(sp.Eq(expr, 0), x)

            integer_roots = []
            for r in roots:
                if r.is_integer:
                    integer_roots.append(int(r))

            if integer_roots:
                return min(integer_roots) % ANSWER_MOD

    except Exception:
        pass

    # Safe fallback answer if no clear equation structure
    return 0

In [10]:
# --- LLM‑based reasoning layer (this is where you can climb score) ---
# High‑score AIMO teams use math‑tuned LLMs (e.g., DeepSeek‑Math‑7B, Qwen‑Math‑7B)
# and chain‑of‑thought + self‑consistency (multiple samples + majority vote).

def load_model(model_name: str = "deepseek‑math‑7b"):
    """
    Load a math‑tuned LLM from Kaggle / Hugging Face.
    Replace `model_name` with a path / model id you've added to Kaggle.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    model.eval()
    return model, tokenizer

In [11]:
def make_math_prompt(problem: str) -> str:
    """
    A CoT prompt template used in many AIMO high‑score entries.
    The model will:
      - think step‑by‑step in natural language
      - optionally compute via Python / symbolic reasoning
      - output the final answer inside \boxed{...}
    """
    return (
        "You are solving an International Mathematical Olympiad‑level problem.\n"
        "Reason step‑by‑step. Use symbolic reasoning or Python‑style computations when helpful.\n"
        "After your reasoning, output the final answer in the format $\\boxed{answer}$.\n\n"
        f"Problem:\n{problem}\n\n"
        "Reasoning:"
    )


In [12]:
def run_llm_once(model, tokenizer, problem: str, device: str) -> str:
    """
    Run one forward pass (one reasoning path) for one problem.
    """
    prompt = make_math_prompt(problem)

    # Tokenize
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=1024,
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
            repetition_penalty=1.1,
        )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Remove the prompt part if present
    if prompt in text:
        text = text[len(prompt):]

    return text

In [13]:
def extract_answer_or_fallback(raw_text: str) -> int:
    """
    Try to extract \boxed{...} first, then fall back to a heuristic.
    """
    box_val = extract_boxed_number(raw_text)
    if box_val is not None:
        return box_val

    # Fallback: try to find the last integer or number in the text
    matches = re.findall(r'-?\d+', raw_text)
    if matches:
        last_num = int(matches[-1])
        return last_num % ANSWER_MOD

    return 0

In [14]:
def solve_with_llm(model, tokenizer, problem: str, device: str, n_samples: int = N_SAMPLES) -> int:
    """
    Use self‑consistency: run multiple reasoning paths, then vote.
    This pattern is widely used in AIMO high‑score solutions.
    """
    answers = []
    for _ in range(n_samples):
        raw_output = run_llm_once(model, tokenizer, problem, device)
        answer = extract_answer_or_fallback(raw_output)
        answers.append(answer)

    # Majority vote (simple one)
    from collections import Counter
    counter = Counter(answers)
    best_answer, _ = counter.most_common(1)[0]
    return best_answer % ANSWER_MOD

In [15]:
# --- Main prediction function (this is your gold‑medal‑style entry point) ---
def predict_answer(problem: str):
    """
    This is your upgraded baseline.
    Replace this function with a stronger solver later (e.g., add more heuristics or verifiers).
    """
    # 1. Try very easy polynomial equations with sympy
    easy_answer = solve_with_sympy_if_easy(problem)
    if easy_answer != 0:
        return easy_answer

    # 2. If available, use the LLM reasoning layer
    #    (for now, this is a placeholder; you can enable it later)
    #    In practice, you will load model/tokenizer once globally
    #    and pass them to this function.
    #
    # Example enabling:
    #   return solve_with_llm(model, tokenizer, problem, "cuda", N_SAMPLES)
    #
    # For now, just return the safe fallback:
    return 0

In [17]:
# --- Build submission DataFrame ---
submission_df = pd.DataFrame()
submission_df["id"] = test_df["id"]

problem_column = None
for candidate in ["problem", "question", "prompt"]:
    if candidate in test_df.columns:
        problem_column = candidate
        break

if problem_column is None:
    raise ValueError(
        "No problem text column was found. "
        "Expected one of: problem, question, prompt"
    )

# Apply your upgraded solver (currently using sympy + fallback)
submission_df["answer"] = test_df[problem_column].apply(predict_answer).astype(int)
submission_df["answer"] = submission_df["answer"].clip(0, ANSWER_MOD - 1)

output_path = "/kaggle/working/submission.csv"
submission_df.to_csv(output_path, index=False)

print("Saved submission file to:", output_path)
submission_df

Saved submission file to: /kaggle/working/submission.csv


,id,answer
0,000aaa,0
1,111bbb,0
2,222ccc,0
